# Real-time Chatterbox + Mistral + Faster-Whisper

Colab prototype for speech → text → streamed response → speech.

In [ ]:
%pip install -q -U chatterbox-tts==0.1.7 faster-whisper
%pip install -q --no-cache-dir --force-reinstall torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -q --no-cache-dir --force-reinstall numpy==1.26.4
%pip install -q -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## Download models

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

CHATTERBOX_MODEL_PATH = snapshot_download(
    repo_id="ResembleAI/chatterbox-turbo"
)

MISTRAL_MODEL_PATH = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q5_0.gguf",
    local_dir="/content/mistral-7b-instruct-v0.2"
)

print("Chatterbox:", CHATTERBOX_MODEL_PATH)
print("Mistral:", MISTRAL_MODEL_PATH)

## Load models

In [ ]:
import torch
from chatterbox.tts_turbo import ChatterboxTurboTTS
from llama_cpp import Llama
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Chatter_model = ChatterboxTurboTTS.from_local(
    CHATTERBOX_MODEL_PATH,
    device=DEVICE
)

llm = Llama(
    model_path=MISTRAL_MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False
)

Whisper_model = WhisperModel(
    "small.en",
    device=DEVICE,
    compute_type="float16" if DEVICE == "cuda" else "int8"
)

print(f"Ready on {DEVICE}: Faster-Whisper + Mistral + Chatterbox")

## Prepare reference voice

Use a clean reference clip longer than 5 seconds.

In [ ]:
from google.colab import files

uploaded = files.upload()
VOICE_FILE = next(iter(uploaded))
Chatter_model.prepare_conditionals(f"/content/{VOICE_FILE}")

print("Voice prepared:", VOICE_FILE)

## Speech input

In [ ]:
from base64 import b64decode
from google.colab import output as colab_output


def listen(max_seconds=15, silence_seconds=1.1, language="en"):
    max_ms = int(max_seconds * 1000)
    silence_ms = int(silence_seconds * 1000)

    javascript = f"""
    (async () => {{
        const stream = await navigator.mediaDevices.getUserMedia({{
            audio: {{echoCancellation: true, noiseSuppression: true, autoGainControl: true}}
        }});
        const recorder = new MediaRecorder(stream);
        const chunks = [];
        const context = new AudioContext();
        const source = context.createMediaStreamSource(stream);
        const analyser = context.createAnalyser();
        analyser.fftSize = 2048;
        const data = new Uint8Array(analyser.fftSize);
        source.connect(analyser);

        recorder.ondataavailable = e => {{ if (e.data.size > 0) chunks.push(e.data); }};
        recorder.start(250);

        const started = Date.now();
        let speechSeen = false;
        let lastSpeech = started;

        while (Date.now() - started < {max_ms}) {{
            analyser.getByteTimeDomainData(data);
            let sum = 0;
            for (const value of data) {{
                const x = (value - 128) / 128;
                sum += x * x;
            }}
            const rms = Math.sqrt(sum / data.length);
            const now = Date.now();

            if (rms > 0.025) {{
                speechSeen = true;
                lastSpeech = now;
            }}

            if (speechSeen && now - lastSpeech >= {silence_ms}) break;
            if (!speechSeen && now - started >= 5000) break;
            await new Promise(resolve => setTimeout(resolve, 100));
        }}

        const stopped = new Promise(resolve => recorder.onstop = resolve);
        recorder.stop();
        await stopped;
        stream.getTracks().forEach(track => track.stop());
        await context.close();

        const blob = new Blob(chunks, {{type: recorder.mimeType || "audio/webm"}});
        return await new Promise(resolve => {{
            const reader = new FileReader();
            reader.onloadend = () => resolve(reader.result);
            reader.readAsDataURL(blob);
        }});
    }})()
    """

    audio_data = colab_output.eval_js(javascript)
    if not audio_data:
        return ""

    audio_file = "/content/user_audio.webm"
    with open(audio_file, "wb") as f:
        f.write(b64decode(audio_data.split(",", 1)[1]))

    segments, _ = Whisper_model.transcribe(
        audio_file,
        language=language,
        beam_size=1,
        vad_filter=True,
        condition_on_previous_text=False
    )

    return " ".join(segment.text.strip() for segment in segments).strip()

## Conversation pipeline

In [ ]:
import queue
import re
import threading
import time
from IPython.display import Audio, display

CHAT_HISTORY = []
HISTORY_TURNS = 6

SYSTEM_PROMPT = """You are a warm, compassionate, biblically grounded Christian theology companion.
Speak like a caring Christian mentor or trusted friend: natural, concise, gentle with people, and firm about biblical truth.
Follow the user's pace. Simple questions deserve simple answers; explain more only when useful.
Base theological answers on Scripture and reliable biblical knowledge. Never invent verses, quotations, doctrines, historical facts, or sources.
When major Christian traditions reasonably disagree, briefly distinguish the main interpretations when relevant.
Be honest about sin, repentance, grace, judgment, holiness, forgiveness, obedience, and faith without condemnation or cruelty.
Do not introduce yourself as an AI unless directly necessary. If you do not know something, say so."""


def reset_chat():
    CHAT_HISTORY.clear()


def build_prompt(user_input):
    history = "\n".join(
        f"User: {turn['user']}\nAssistant: {turn['assistant']}"
        for turn in CHAT_HISTORY[-HISTORY_TURNS:]
    )

    conversation = f"Conversation so far:\n{history}\n\n" if history else ""
    return f"[INST] {SYSTEM_PROMPT}\n\n{conversation}User: {user_input.strip()}\nRespond naturally to the user. [/INST]"


def take_phrase(buffer, max_chars=180):
    match = re.search(r"^(.+?[.!?])(?=\s|$)", buffer, flags=re.S)
    if match:
        end = match.end()
        return buffer[:end].strip(), buffer[end:].lstrip()

    if len(buffer) < max_chars:
        return None, buffer

    cuts = [buffer.rfind(mark, 0, max_chars) for mark in [",", ";", ":"]]
    cut = max(cuts)
    if cut < 60:
        cut = buffer.rfind(" ", 0, max_chars)
    if cut < 1:
        cut = max_chars - 1

    return buffer[:cut + 1].strip(), buffer[cut + 1:].lstrip()


def generate_response_stream(user_input, max_tokens=220):
    if not user_input or not user_input.strip():
        return ""

    text_queue = queue.Queue()
    audio_queue = queue.Queue()
    errors = []
    stop_signal = object()

    def tts_worker():
        try:
            while True:
                phrase = text_queue.get()
                if phrase is stop_signal:
                    break
                wav = Chatter_model.generate(phrase).detach().cpu().squeeze()
                audio_queue.put(wav)
        except Exception as e:
            errors.append(e)
        finally:
            audio_queue.put(stop_signal)

    def audio_worker():
        try:
            while True:
                wav = audio_queue.get()
                if wav is stop_signal:
                    break
                display(Audio(wav.numpy(), rate=Chatter_model.sr, autoplay=True))
                time.sleep(wav.shape[-1] / Chatter_model.sr + 0.05)
        except Exception as e:
            errors.append(e)

    tts_thread = threading.Thread(target=tts_worker)
    audio_thread = threading.Thread(target=audio_worker)
    tts_thread.start()
    audio_thread.start()

    full_response = []
    speech_buffer = ""
    print("Assistant: ", end="", flush=True)

    try:
        response = llm(
            build_prompt(user_input),
            max_tokens=max_tokens,
            stop=["</s>", "\nUser:"],
            stream=True
        )

        for packet in response:
            text = packet["choices"][0]["text"]
            if not text:
                continue

            print(text, end="", flush=True)
            full_response.append(text)
            speech_buffer += text

            while True:
                phrase, speech_buffer = take_phrase(speech_buffer)
                if phrase is None:
                    break
                text_queue.put(phrase)

        if speech_buffer.strip():
            text_queue.put(speech_buffer.strip())
    finally:
        text_queue.put(stop_signal)
        tts_thread.join()
        audio_thread.join()

    if errors:
        raise errors[0]

    answer = "".join(full_response).strip()
    CHAT_HISTORY.append({"user": user_input.strip(), "assistant": answer})
    del CHAT_HISTORY[:-HISTORY_TURNS]
    return answer

## Start voice conversation

Run this cell and speak normally. The recorder stops after you go quiet. Use **Ctrl+C** to stop.

In [ ]:
while True:
    try:
        print("\nListening...")
        user_text = listen(max_seconds=15, silence_seconds=1.1, language="en")

        if not user_text:
            print("No speech detected.")
            continue

        print("User:", user_text)
        generate_response_stream(user_text, max_tokens=220)
        print()

    except KeyboardInterrupt:
        print("\nStopped.")
        break
    except Exception as e:
        print(f"\nError: {e}")
        break